# Phase 2: Negative review classification

**Question:** Will a delivered order receive a low customer review (1–2 stars)?

**Stakeholder:** Customer experience / seller quality team. Predictions are made at delivery, before a review exists, to prioritise orders at risk.

**Scope:** Two model families only: Linear (Logistic Regression) and tree-based (Decision Tree, Random Forest). This notebook follows the IT5006 Phase 2 checklist. It reuses the project's existing feature and pipeline functions without changing them. The [original combined notebook](phase2_delivered_order_classification_combined.ipynb) remains untouched.

1. Data preparation and target
2. Leakage check and chronological train/test split
3. Simple models and cross-validation
4. Hyperparameter tuning
5. Held-out evaluation
6. Model interpretation
7. Error analysis and stakeholder conclusion

No separate intervention system is modelled here. Filtering previously reviewed orders is **preprocessing**, not an additional modelling stage.

Data inputs are local Phase 1 processed CSVs in `data/processed/` (excluded from Git).

## 1. Data preparation and target

One record per delivered order with a known review. Target: **1** for 1–2 stars and **0** for 3–5 stars. The 21 candidate features include delivery performance, order values, product details and geography. These features are available at delivery.

**Filter:** Exclude orders reviewed **before** delivery. For those orders the proposed prediction would come too late. Review timestamps are used to filter the historical training population, never as model predictors.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit, cross_validate, GridSearchCV

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.classification_data import (
    TARGET_COLUMN, DELIVERED_MODEL_FEATURES, build_delivered_order_classification_dataset,
    split_delivered_order_features_target,
)
from src.classification_model import build_delivered_order_logistic_pipeline
from src.classification_tree_model import (
    build_delivered_order_decision_tree_pipeline,
    build_delivered_order_random_forest_pipeline,
)

SEED = 42
orders = pd.read_csv(ROOT / "data/processed/order_level.csv")
items = pd.read_csv(ROOT / "data/processed/item_level.csv")
df = build_delivered_order_classification_dataset(orders, items)
lookup = orders.set_index("order_id").loc[df["order_id"]]
df["purchase_time"] = pd.to_datetime(lookup["order_purchase_timestamp"], format="mixed").to_numpy()
reviewed_at = pd.to_datetime(lookup["review_answer_timestamp"], format="mixed").to_numpy()
delivered_at = pd.to_datetime(lookup["order_delivered_customer_date"], format="mixed").to_numpy()
already_reviewed = reviewed_at < delivered_at
print(f"Delivered orders with review: {len(df):,}")
print(f"Excluded (review before delivery): {already_reviewed.sum():,}")
df = df.loc[~already_reviewed].sort_values("purchase_time", kind="stable").reset_index(drop=True)
print(f"Eligible orders: {len(df):,}; negative review rate: {df[TARGET_COLUMN].mean():.1%}")
display(df[TARGET_COLUMN].value_counts().rename(index={0:"3–5 stars",1:"1–2 stars"}).to_frame("orders"))

## 2. Leakage and chronological split

The model scores newly delivered orders **before** they submit a review. Delivery-related inputs are allowed at this prediction point; review score defines the target only. Review content, score and timestamps are not predictors. A chronological 80/20 split by **purchase date** holds out later orders for an unbiased final evaluation; purchase date is a practical proxy, not a full event-time production simulation.

Imputation, scaling and encoding are performed within the existing scikit-learn pipelines, fitted on each training fold. No preprocessing statistics are fitted on the held-out set.

In [ ]:
X, y = split_delivered_order_features_target(df)
assert X.columns.tolist() == DELIVERED_MODEL_FEATURES
assert not any("review" in col for col in X.columns)
cut = int(len(df) * 0.80)
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
y_train, y_test = y.iloc[:cut], y.iloc[cut:]
display(pd.DataFrame([
    {"population": "Train", "orders": len(y_train), "negative_rate": y_train.mean(),
     "first_purchase": df["purchase_time"].iloc[0].date(),
     "last_purchase": df["purchase_time"].iloc[cut-1].date()},
    {"population": "Test", "orders": len(y_test), "negative_rate": y_test.mean(),
     "first_purchase": df["purchase_time"].iloc[cut].date(),
     "last_purchase": df["purchase_time"].iloc[-1].date()},
]).round(3))

## 3. Simple baselines and time-based cross-validation

Start with the simplest version in each family: plain Logistic Regression and a single Decision Tree. Then compare a default Random Forest before tuning. This demonstrates whether complexity is justified.

We use five chronological CV folds, all within training data. **PR AUC (average precision)** is the primary selection metric because low reviews are relatively rare. Precision, recall, F1 and ROC AUC are also reported. Threshold metrics below use the default 0.50 cutoff; ROC AUC and PR AUC assess ranking across thresholds.

In [ ]:
cv = TimeSeriesSplit(n_splits=5)
metrics = {"precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc","pr_auc":"average_precision"}
baseline = {
    "Logistic (plain)": build_delivered_order_logistic_pipeline(random_state=SEED).set_params(classifier__C=np.inf),
    "Decision Tree (plain)": build_delivered_order_decision_tree_pipeline(random_state=SEED),
    "Random Forest (default)": build_delivered_order_random_forest_pipeline(random_state=SEED, n_estimators=200),
}
rows = []
for name, pipeline in baseline.items():
    result = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=metrics, n_jobs=1)
    rows.append({"model":name, **{metric:result[f"test_{metric}"].mean() for metric in metrics}})
baseline_cv = pd.DataFrame(rows).set_index("model")
display(baseline_cv.round(3))

## 4. Tune Logistic Regression and Random Forest

Use limited grids for the **same two families**. Select parameters using training CV **PR AUC**, never the held-out test results. Logistic Regression varies regularisation and class weighting; Random Forest varies depth, leaf size and class weighting. The search is deliberately small to focus on well-justified comparisons.

In [ ]:
logistic_grid = GridSearchCV(
    build_delivered_order_logistic_pipeline(random_state=SEED),
    {"classifier__C":[0.01,0.1,1.0,10.0],
     "classifier__class_weight":[None,"balanced"]},
    cv=cv, scoring=metrics, refit="pr_auc", n_jobs=1,
)
forest_grid = GridSearchCV(
    build_delivered_order_random_forest_pipeline(random_state=SEED,n_estimators=200),
    {"classifier__max_depth":[12,None],
     "classifier__min_samples_leaf":[10,25],
     "classifier__class_weight":[None,"balanced_subsample"]},
    cv=cv, scoring=metrics, refit="pr_auc", n_jobs=1,
)
logistic_grid.fit(X_train,y_train)
forest_grid.fit(X_train,y_train)
tuned = {"Logistic (tuned)":logistic_grid, "Random Forest (tuned)":forest_grid}
print("Selected parameters (training CV only):")
for name, search in tuned.items():
    print(name, search.best_params_)
tuned_cv = pd.DataFrame([
    {"model":name, **{metric:search.cv_results_[f"mean_test_{metric}"][search.best_index_] for metric in metrics}}
    for name,search in tuned.items()
]).set_index("model")
cv_results = pd.concat([baseline_cv,tuned_cv])
display(cv_results.sort_values("pr_auc",ascending=False).round(3))
chosen_name = cv_results["pr_auc"].idxmax()
print("Preselected model using CV PR AUC:",chosen_name)

## 5. Test performance on later orders

Fit each untuned baseline to the full training set; tuned searches already refit their chosen models on training data. Evaluate all on the previously untouched test set. **Do not re-tune or change the selection rule after seeing these results.**

A model may rank negative reviews reasonably well (PR AUC/ROC AUC) but predict no positive cases at the 0.50 cutoff. This is a threshold issue, not necessarily a ranking failure.

In [ ]:
fitted = {name:clone(model).fit(X_train,y_train) for name,model in baseline.items()}
fitted.update({name:search.best_estimator_ for name,search in tuned.items()})
probs, test_rows = {}, []
for name,model in fitted.items():
    p = model.predict_proba(X_test)[:,1]
    pred = (p>=0.5).astype(int)
    probs[name] = p
    test_rows.append({
        "model":name, "precision":precision_score(y_test,pred,zero_division=0),
        "recall":recall_score(y_test,pred,zero_division=0),
        "f1":f1_score(y_test,pred,zero_division=0),
        "roc_auc":roc_auc_score(y_test,p), "pr_auc":average_precision_score(y_test,p)
    })
test_results = pd.DataFrame(test_rows).set_index("model")
display(test_results.round(3))
print(f"Test prevalence, the random ranking PR reference: {y_test.mean():.3f}")
print("Model selected before opening the test set:",chosen_name)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for name in tuned:
    fpr,tpr,_ = roc_curve(y_test,probs[name])
    precision,recall,_ = precision_recall_curve(y_test,probs[name])
    axes[0].plot(fpr,tpr,label=name)
    axes[1].plot(recall,precision,label=name)
axes[0].plot([0,1],[0,1],linestyle=":",label="Random")
axes[1].axhline(y_test.mean(),linestyle=":",label="Prevalence")
axes[0].set(title="ROC",xlabel="False positive rate",ylabel="True positive rate")
axes[1].set(title="Precision–recall",xlabel="Recall",ylabel="Precision")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Feature importance and interpretation

Permutation importance estimates how much the **held-out PR AUC drops** when one feature is shuffled. Treat this as evidence of model reliance, **not** causal effects. Results from this diagnostic are for interpretation, not for revising the model after testing.

In [ ]:
importance = permutation_importance(
    fitted[chosen_name], X_test, y_test, scoring="average_precision",
    n_repeats=3, random_state=SEED, n_jobs=1
)
feature_importance = pd.Series(
    importance.importances_mean,index=X_test.columns,name="PR AUC decrease"
).sort_values(ascending=False)
display(feature_importance.head(10).round(4).to_frame())

## 7. Error analysis

The confusion matrix displays misses and false alarms for the **illustrative 0.50 threshold**. The business might later choose a different threshold or a limited contact queue, but that would need its own explicit capacity and validation policy.

In [ ]:
chosen_prob = probs[chosen_name]
pred = (chosen_prob>=0.5).astype(int)
display(pd.DataFrame(
    confusion_matrix(y_test,pred,labels=[0,1]),
    index=["Actual 3–5 stars","Actual 1–2 stars"],
    columns=["Predicted 3–5 stars","Predicted 1–2 stars"],
))
errors = X_test.copy()
errors["actual"] = y_test.to_numpy()
errors["predicted"] = pred
errors["result"] = np.select(
    [(errors["actual"]==1)&(errors["predicted"]==0),
     (errors["actual"]==0)&(errors["predicted"]==1)],
    ["Missed negative","False alarm"],default="Correct",
)
display(errors.groupby("result").agg(
    orders=("actual","size"), late_share=("late_delivery_flag","mean"),
    median_delivery_days=("delivery_days","median"),
).round(3))

## 8. Conclusion, stakeholder use and limitations

**Final model:** `chosen_name` is determined from *training CV PR AUC*, not from the test set. In your Phase 2 report, present the two tuned finalists' CV and held-out metrics, note whether any gain justifies the extra complexity, and justify the final choice. A similar held-out result does not prove the models are identical.

**Practical use:** A customer experience team could prioritise follow-up for newly delivered orders that have not already received reviews. A predicted risk score does not prove an intervention will prevent a poor review.

**Limitations:** Only historical Olist orders are available; outcomes were not known at deployment time; pre-delivery reviews have been filtered; customer dissatisfaction has causes missing from the available features; and purchase-time splitting does not simulate every production timing detail.

**Phase 3:** This notebook does not modify or overwrite existing model artifacts. Agree the final model and deployment feature contract before saving a retrained pipeline.

See the [original combined notebook](phase2_delivered_order_classification_combined.ipynb) for the optional operational exploration and additional diagnostics.